

```
# This is formatted as code
```

# Zen Market New Feature Proposal: Parcel cost prediction and product upsell engine

**Concept:** Use existing historical Zen Market data and available third party data to create a machine learning model that predicts parcel size, and uses that sizing to estemate shipping costs. Using that figure, we then add a banner to the site that advertises savings for the user if they were to add additional items.

We can then add in email marketing centered around reccomending items that were purchased by other customers who also purchased the same items.

## 1. The problem

The current Zen Market site requires that the user enter parcel size to get a shipping estemate. This creates a barrier to customers who might be hestant to purchase an item with an unknown final price.

But once a customer has an item in their cart, on the way to the warehouse, or waiting international fulfilment, they would likely be more inclined to order subsequent items if they are presented with the marginal shipping cost of additional items.

This is a key diferentiator between Zen Market and its competators like Buyee and FROM JAPAN that will likely entice customers to favor Zen Market, and encourage customers to purchase more than they might have originally.

## 2. Proposed UI

A banner appears at the top of the page that changes depending on the stage of the customers purchase:

- **Just Browsing, no purchases or items in the cart:** “Estimated international shipping for this item: ¥2,000–¥2,700.”
- **Shopping with items in the cart, in transit, or at the warehouse:** “Adding this item could save you up to ¥2,000 in shipping costs!”

Optional email notifications suggest items other users who purchased that same item also purchaed, along with the cost savings for consolidating the items shipping with an existing order.

## 4. Data needed

| Data group | Example fields | Primary role |
|---|---|---|
| Product listings | Title, Japanese description, category, photos, price, seller | Identify similar goods and product properties |
| Measured items | Actual item weight, measured dimensions where available | Train and validate item estimates |
| Completed parcels | Contents, number of items, final packaged weight, outer dimensions | Learn consolidation/packing effects |
| Shipping transactions | Destination, service, charged rate, date, surcharges | Validate estimated charges and segment outcomes |
| Current shipping rules | Rate tables, volumetric weight divisor, prohibited items, limits | Calculate eligible quotes using current rules |
| Shopping signals | Consented purchases, favorites, searches, recent views | Personalized recommendations |

## 5. Prediction methodology

### A. Build an item property estimator
1. Standardize listing titles and categories across Japanese and other languages.
2. Link listings to historical items with reliable measurements; deduplicate and exclude suspicious records.
3. Identify feature set: product size, weights, quantities, etc.
5. Compare these with a tabular regression model such as LightGBM or ML.NET models. Predict weight and dimensions using uncertainty intervals.

### B. Estimate packed parcel properties
1. Use warehouse item data with known dimensions/weight when available.
2. For a newly proposed item we predict weight and dimensions based on like items using the ML model.
3. Estemate consolidated packing with historical packing datta, packaging allowances, incompatibility rules, and dimension/weight constraints.
4. Estimate the distribution of possible packed weights and dimensions instead of assuming that volume or dimensions can be always be neatly packaged.

### C. Apply current shipping prices
Feed the estimated measurements into deterministic carrier rate logic or authorized quotation APIs, applying destination, service availability, dimensions, volumetric weight, taxes/surcharges where in scope, and packing restrictions.

The ML model predicts dimensions and weights while the pricing layer calculates rates

## 6. Shipping comparison example


| Scenario | Estimate |
|---|---:|
| Existing parcel A | ¥3,200 |
| New item B shipped separately | ¥2,500 |
| A+B shipped together | ¥3,650 |
| Marginal cost to add B | **¥450** |
| Savings versus separate shipments | **¥2,050** |

The system should show both figures; For high uncertainty cases, use ranges and limit unrealistic claims that are not realistic.

## 7. Personalization and relevance

Generate reccomended items using historical customer data, product metadata, and semantic similarity across listings.

We would then rank candidates using a combination of:

- **Interest:** probability the customer would purchase regardless of shipping savings.
- **Incremental shipping cost:** predicted additional shipping cost.
- **Consolidation advantage:** predicted saving compared with separate shipment.
- **Eligibility and reliability:** whether the product can be shipped together.

This avoids reccomending items for the sake of doing so and being percieved as spam. The intention should be for this to be a value add for the customer.

## 8. Design Stack

| Layer | Proposed choice | Responsibilities |
|---|---|---|
| Frontend | React + TypeScript | Persistent banner, comparison card, recommendations, loading/error states |
| API/backend | ASP.NET Core (C#) | User/cart context, orchestration, shipping logic, consent, REST endpoints |
| Persistence | SQL Server + Entity Framework Core | Product history, measured items, parcel contents, estimates, audit records |
| Predictive model | ML.NET initially; Python/LightGBM for experiments | Weight and dimension estimates, confidence intervals |
| Recommendations | Semantic text embeddings plus deterministic reranking | Cross language matching and shipping aware product ranking |
| Notifications | Scheduled worker + transactional email provider | Opt in, frequency limited alerts |
| Testing/telemetry | xUnit, API integration tests, OpenTelemetry | Correctness, latency, error monitoring |


## 9. Evaluation

Run an A/B test on eligible customers:

- **Control:** existing shopping experience / conventional recommendations.
- **Treatment:** predictive shipping banner plus shipping aware recommendations.

Intended outcome: noted increase in gross profit per eligible user or purchase conversion.

Secondary outcomes: increase in average basket value, items purchased, and repeat visits.

## 10. Risks and safeguards

| Risk | Mitigation |
|---|---|
| Incorrect historical item to parcel attribution | Use measured item level labels or explicitly modeled estimates |
| Packing geometry and fragile/incompatible items | Eligibility rules, uncertainty, conservative packing estimates |
| Carrier pricing changes | Versioned, current rate data separate from ML |
| Misleading “savings” messaging | Compare genuine separate and combined scenarios; show total cost |
| Cross border restrictions | Restriction checks and manual review; no automatic eligibility assumption |
| Sensitive purchase histories | Consent, retention limits, minimal exposure, access controls |
| Biased or sparse categories | Category level baselines and abstention when confidence is low |
| Intrusive email recommendations | Opt-in, unsubscribe, frequency caps |

## 11. Demonstration walkthrough

1. Open the prototype and visit a listing with no seller provided weight.
3. The model estimates its likely item weight, packing dimensions, and shipping price range.
4. The banner automatically compares the existing parcel against a combined shipment.
5. A related, personalized product appears with a marginal shipping estimate and separate vs combined cost comparison.
6. Add it to the cart and UI estimates update immediately.

## 12. Proposed development stages

**Phase 1 ** — Identify what data is available, and how we can strenghten data collection and feature evaluation.

**Phase 2 ** Implement category/nearest neighbor baseline, validate on held out items, and add uncertainty intervals.

**Phase 3 ** Add packing assumptions, carrier rules, marginal cost calculation, and persistent contextual banner.

**Phase 4 ** Reproducible prediction tests, rate calculation regression tests, and a proposed A/B test instrumentation plan.